# Hidden Sardinian Gems — mappa dinamica 2022-2025

Notebook rerunnable end-to-end. Parte dai **16 CSV di indicatori comunali** (densità turistica,
intensità turistica, densità ricettiva, indice di overtourism × 4 anni), ricostruisce il punteggio
gemma per ogni comune e per ognuno dei 48 mesi, e produce una mappa HTML autosufficiente con
timeline scorrevole.

## Che cosa è cambiato rispetto alla versione precedente

| Problema | Correzione |
|---|---|
| Leggeva i CSV da `/workspace/tmp`, la geometria da un file locale e delegava il calcolo a tre script esterni (`fetch_osm_data.py`, `compute_attractiveness.py`, `build_data_with_osm.py`) mai inclusi | Tutta la pipeline è dentro il notebook; CSV e `comuni_sardegna.geojson` sono gli input, nessuna dipendenza di rete |
| L'attrattività veniva da POI OpenStreetMap via Overpass | Overpass non è raggiungibile e i CSV OSM pre-computati non esistono: l'indice è ricostruito dai CSV di presenze. **Le due cose non misurano lo stesso fenomeno** — vedi la sezione 4 |
| Il ranking era calcolato per il solo agosto 2025 | Il punteggio è calcolato per tutti i 377 comuni × 48 mesi |
| `marine_gem_bonus` veniva da un database curato dentro `data.json` | Sostituito da un flag costiero ricavato geometricamente dai confini comunali |
| Eleggibilità e stagionalità erano citate nel testo ma non implementate in nessuna cella | Implementate e verificate esplicitamente |
| `matplotlib.use('Agg')` con `plt.tight_layout()` e nessun `savefig`: i grafici non venivano mai prodotti | I grafici vengono salvati su file |


## 1. Ambiente e parametri

Tutti i parametri del modello stanno qui, così una modifica si propaga a valle senza toccare il codice.

In [ ]:
import json, unicodedata, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from shapely.geometry import shape, mapping
from shapely.ops import unary_union

# --- percorsi: il primo che esiste vince, così il notebook gira sia in locale sia nel container
CSV_DIR = next(p for p in [Path('/mnt/user-data/uploads'), Path('/workspace/tmp'), Path('.')]
               if (p / 'indice_overtourism_2025.csv').exists())
OUT = Path('/mnt/user-data/outputs') if Path('/mnt/user-data').exists() else Path('./out')
OUT.mkdir(parents=True, exist_ok=True)

YEARS   = [2022, 2023, 2024, 2025]
MESI    = ['gennaio','febbraio','marzo','aprile','maggio','giugno',
           'luglio','agosto','settembre','ottobre','novembre','dicembre']

# --- pilastri dell'attrattività rivelata e relativi pesi
PILLARS  = ['permanenza', 'continuita', 'destagionalita']
WEIGHTS  = {'permanenza': 0.40, 'continuita': 0.25, 'destagionalita': 0.35}

# --- moltiplicatori e soglie del gem score
MARINE_BONUS     = 0.15   # bonus ai comuni costieri
SEASONAL_MAX     = 0.50   # correzione massima nel mese più lontano dal picco
MIN_ARRIVI_ANNO  = 100    # guardia contro i comuni con volumi da rumore statistico
OVERTOURISM_Q    = 0.75   # percentile di esclusione, calcolato sull'intero panel
SIMPLIFY_DEG     = 0.0008 # semplificazione delle geometrie per l'HTML (~70 m)
COAST_TOL_DEG    = 0.002  # lunghezza minima di confine condiviso con la costa

assert sum(WEIGHTS.values()) == 1.0
print('CSV:', CSV_DIR, '| output:', OUT)

## 2. I sedici CSV

Ogni famiglia di indicatori arriva con una griglia diversa: `indice_overtourism` e
`densita_ricettiva` coprono tutti i 377 comuni × 12 mesi × 4 anni (18 096 righe), mentre
`densita_turistica` e `intensita_turistica` esistono solo dove ISTAT registra presenze
(12 984 righe). Il merge è `left` sull'overtourism, che è la griglia completa.

In [ ]:
def load(prefix):
    frames = []
    for y in YEARS:
        f = CSV_DIR / f'{prefix}_{y}.csv'
        df = pd.read_csv(f, encoding='utf-8-sig')
        assert {'comune', 'anno', 'mese'} <= set(df.columns), f
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

overtourism = load('indice_overtourism')
intensita   = load('intensita_turistica')
ricettiva   = load('densita_ricettiva')
densita     = load('densita_turistica')

for nome, df in [('indice_overtourism', overtourism), ('intensita_turistica', intensita),
                 ('densita_ricettiva', ricettiva), ('densita_turistica', densita)]:
    print(f'{nome:22s} {len(df):>6} righe  {df.comune.nunique():>3} comuni')

panel = (overtourism
    .merge(intensita[['comune','anno','mese','popolazione_residente','presenze_mese','arrivi_mese']],
           on=['comune','anno','mese'], how='left')
    .merge(ricettiva[['comune','anno','mese','letti_totali','superficie_kmq']],
           on=['comune','anno','mese'], how='left')
    .merge(densita[['comune','anno','mese','presenze_stimate']],
           on=['comune','anno','mese'], how='left'))

panel['presenze_mese']    = panel.presenze_mese.fillna(0.0)
panel['arrivi_mese']      = panel.arrivi_mese.fillna(0.0)
panel['presenze_stimate'] = panel.presenze_stimate.fillna(False).astype(bool)

assert len(panel) == 377 * 12 * len(YEARS) == 18096
assert panel.duplicated(['comune','anno','mese']).sum() == 0
print('\nPanel:', panel.shape, '| comuni', panel.comune.nunique())

### 2b. Copertura dei dati — l'avvertenza più importante del notebook

I comuni con dati di presenza passano da 189 nel 2022 a 322 nel 2025. Una parte consistente
dell'aumento di gemme fra il 2022 e il 2025 è **crescita della copertura statistica**, non un
cambiamento reale del turismo sardo. Il confronto fra anni diversi va letto con questa cautela;
il confronto fra mesi dello stesso anno è invece pulito.

In [ ]:
cop = pd.DataFrame({
    'comuni con presenze': intensita.groupby('anno').comune.nunique(),
    'righe con presenze > 0': intensita[intensita.presenze_mese > 0].groupby('anno').size(),
    'presenze stimate (non rilevate)': densita[densita.presenze_stimate == True].groupby('anno').size(),
})
print(cop.to_string())

## 3. Geometria comunale e flag costiero

I 377 poligoni arrivano da `comuni_sardegna.geojson` (limiti ISTAT, schema openpolis), che è ora
un input del notebook allo stesso titolo dei CSV: nessun download, la pipeline gira offline.
Il notebook cerca il file fra i percorsi noti e ricade sullo scaricamento solo se non lo trova.

Sciogliendo i poligoni con `unary_union` si ottiene il profilo dell'isola: un comune è costiero
se il suo confine condivide con quel profilo più di `COAST_TOL_DEG` di lunghezza.

I nomi dei comuni nei CSV differiscono dal geojson per sei casi di maiuscole nelle preposizioni
(`Alà Dei Sardi` contro `Alà dei Sardi`): il match si fa su una chiave normalizzata.

In [ ]:
GEO_CANDIDATES = [CSV_DIR / 'comuni_sardegna.geojson',
                  Path('comuni_sardegna.geojson'),
                  OUT / 'comuni_sardegna.geojson']
GEO_PATH = next((p for p in GEO_CANDIDATES if p.exists()), None)

if GEO_PATH is None:   # ripiego di rete, usato solo se il file non è fra gli input
    GEO_PATH = OUT / 'comuni_sardegna.geojson'
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/openpolis/geojson-italy/master/'
        'geojson/limits_R_20_municipalities.geojson', GEO_PATH)
    print('geojson non trovato fra gli input: scaricato in', GEO_PATH)
else:
    print('geometria:', GEO_PATH)

geo = json.loads(GEO_PATH.read_text(encoding='utf-8'))
assert geo['type'] == 'FeatureCollection' and len(geo['features']) == 377
assert all({'name','com_istat_code','prov_name'} <= set(f['properties']) for f in geo['features'])
assert len({f['properties']['com_istat_code'] for f in geo['features']}) == 377

shapes = {f['properties']['name']: shape(f['geometry']) for f in geo['features']}
istat  = {f['properties']['name']: f['properties']['com_istat_code'] for f in geo['features']}
prov   = {f['properties']['name']: f['properties']['prov_name'] for f in geo['features']}

land    = unary_union(list(shapes.values()))
coastal = {n: bool(g.boundary.intersection(land.boundary).length > COAST_TOL_DEG)
           for n, g in shapes.items()}

def norm(s):
    s = unicodedata.normalize('NFKD', str(s))
    return ''.join(c for c in s if not unicodedata.combining(c)).lower().replace("'", '').replace(' ', '')

name_by_key = {norm(n): n for n in shapes}
assert set(map(norm, panel.comune.unique())) == set(name_by_key), 'nomi comune non allineati'

bbox = land.bounds
print('Comuni costieri:', sum(coastal.values()), '/ 377')
print('Estensione: lon %.3f–%.3f  lat %.3f–%.3f' % (bbox[0], bbox[2], bbox[1], bbox[3]))
print('Controllo:', {c: coastal[c] for c in ['Alghero','Villasimius','Nuoro','Macomer','Carloforte']})

## 4. Indice di attrattività

**Questa è la sostituzione da tenere presente.** La versione precedente calcolava l'attrattività
dai punti di interesse OpenStreetMap classificati in cinque pilastri (turismo, natura,
ristorazione, servizi, infrastrutture). Quel dato richiede una chiamata a Overpass e non è
ricostruibile qui; i CSV pre-computati non sono fra i file di partenza.

Al suo posto c'è un indice di **attrattività rivelata**, calcolato ogni anno sui soli CSV
disponibili. La differenza è sostanziale e va dichiarata: l'indice OSM misurava *che cosa c'è in
un comune*, questo misura *come si comportano le persone che ci vanno*.

Tre pilastri, tutti indipendenti dal volume assoluto di turisti:

| Pilastro | Peso | Definizione |
|---|---|---|
| Permanenza | 40% | presenze / arrivi nell'anno, tagliato a 15 notti |
| Continuità | 25% | mesi con presenze > 0, su 12 |
| Destagionalità | 35% | quota di presenze annue fuori da luglio-agosto |

Normalizzazione min-max sull'intero panel 2022-2025 (non anno per anno), così un comune che
migliora nel tempo vede salire il proprio indice invece di restare fermo per costruzione.
Winsorizzazione al 99° percentile per non far dettare la scala a un singolo outlier.

La cella successiva lascia comunque il gancio per i CSV OSM: se compaiono, il notebook li usa.

In [ ]:
osm_index_path = OUT / 'indice_attrattivita.csv'   # gancio per l'indice OSM originale

annuale = (intensita.groupby(['comune','anno'])
    .agg(arrivi=('arrivi_mese','sum'), presenze=('presenze_mese','sum'),
         mesi_attivi=('presenze_mese', lambda s: int((s > 0).sum())),
         popolazione=('popolazione_residente','first'))
    .reset_index())
picco = (intensita[intensita.mese.isin([7, 8])]
         .groupby(['comune','anno']).presenze_mese.sum()
         .rename('presenze_luglio_agosto').reset_index())
annuale = annuale.merge(picco, on=['comune','anno'], how='left').fillna({'presenze_luglio_agosto': 0.0})

annuale['permanenza']     = np.where(annuale.arrivi > 0, annuale.presenze / annuale.arrivi, 0).clip(0, 15)
annuale['continuita']     = annuale.mesi_attivi / 12 * 100
annuale['destagionalita'] = np.where(annuale.presenze > 0,
                                     1 - annuale.presenze_luglio_agosto / annuale.presenze, 0) * 100

for p in PILLARS:
    x = annuale[p].clip(upper=annuale[p].quantile(0.99))
    annuale[p + '_n'] = (x - x.min()) / (x.max() - x.min()) * 100

if osm_index_path.exists():
    osm = pd.read_csv(osm_index_path)
    annuale = annuale.merge(osm[['comune','indice']].rename(columns={'indice':'attrattivita'}),
                            on='comune', how='left')
    print('Attrattività: indice OSM caricato da', osm_index_path.name)
else:
    annuale['attrattivita'] = sum(annuale[p + '_n'] * WEIGHTS[p] for p in PILLARS)
    print('Attrattività: indice rivelato dai CSV (nessun file OSM presente)')

print(annuale.attrattivita.describe().round(2).to_string())

### 4b. L'indice è davvero indipendente dall'overtourism?

Il gem score moltiplica `(1 − overtourism)` per l'attrattività. Se i due termini misurassero la
stessa cosa con segno opposto, il punteggio sarebbe degenere. Vale la pena verificarlo invece di
darlo per scontato.

Una prima versione dell'indice includeva anche il richiamo (arrivi/residenti), la dotazione di
posti letto e l'utilizzazione lorda: con quei pilastri la correlazione di rango con l'overtourism
era **0,93**, cioè il punteggio si limitava a girare al contrario la classifica dell'affollamento.
Il richiamo e la dotazione sono misure di volume, e il volume è esattamente quello che
l'overtourism misura; l'utilizzazione lorda è addirittura una delle quattro componenti con cui
l'indice di overtourism è costruito nei CSV di partenza. Rimossi i tre pilastri, la correlazione
scende a circa **0,28**.

In [ ]:
ovt_annuo = overtourism.groupby(['comune','anno']).indice_overtourism.mean().reset_index()
diag = annuale.merge(ovt_annuo, on=['comune','anno'])

print(f"attrattività vs overtourism   pearson {diag.attrattivita.corr(diag.indice_overtourism):+.3f}"
      f"   spearman {diag.attrattivita.corr(diag.indice_overtourism, method='spearman'):+.3f}")
for p in PILLARS:
    print(f"  {p:16s}            spearman "
          f"{diag[p + '_n'].corr(diag.indice_overtourism, method='spearman'):+.3f}")

### 4c. Il limite che i dati non permettono di superare

Le statistiche ISTAT di presenze e arrivi non distinguono soggiorno di lavoro e soggiorno di
vacanza. Alcuni comuni industriali — Sarroch con la raffineria, Uta con la sua zona industriale —
mostrano permanenze lunghissime distribuite su tutto l'anno e finiscono in alto nell'attrattività
per motivi che con il turismo non c'entrano nulla.

Il filtro sull'overtourism ne esclude una parte, ma non è una soluzione: è un effetto collaterale.
La cella qui sotto li rende visibili, e nella mappa il dettaglio dei pilastri permette di
riconoscerli (permanenza al massimo, destagionalità quasi totale).

In [ ]:
sospetti = annuale[(annuale.anno == 2025) & (annuale.arrivi >= 500)].nlargest(10, 'attrattivita')
print(sospetti[['comune','attrattivita','permanenza','continuita','destagionalita','arrivi']]
      .round(1).to_string(index=False))

## 5. Correzione stagionale e bonus costiero

La correzione stagionale premia il mese più lontano dal picco **del singolo comune**, non dal
picco regionale: un comune che si riempie a maggio riceve la correzione ad agosto, non viceversa.
Va da 0 nel mese di picco a `SEASONAL_MAX` nel mese in cui le presenze sono nulle.

Il bonus costiero sostituisce il database curato di grotte e spiagge del `data.json` originale,
che non è fra i file di partenza. È una proxy geometrica, non la stessa cosa.

In [ ]:
panel = panel.merge(
    annuale[['comune','anno','arrivi','attrattivita'] + [p + '_n' for p in PILLARS]],
    on=['comune','anno'], how='left')
panel[['arrivi','attrattivita']] = panel[['arrivi','attrattivita']].fillna(0.0)

panel['picco_comune'] = panel.groupby(['comune','anno']).presenze_mese.transform('max')
panel['seasonal_adjustment'] = np.where(
    panel.picco_comune > 0, (1 - panel.presenze_mese / panel.picco_comune) * SEASONAL_MAX, 0.0)

bonus_lut = {norm(n): (MARINE_BONUS if coastal[n] else 0.0) for n in shapes}
panel['marine_bonus'] = panel.comune.map(lambda c: bonus_lut[norm(c)])

assert panel.seasonal_adjustment.between(0, SEASONAL_MAX).all()
print('correzione stagionale  media %.3f  max %.2f' % (panel.seasonal_adjustment.mean(),
                                                       panel.seasonal_adjustment.max()))
print('righe con bonus costiero:', int((panel.marine_bonus > 0).sum()))

## 6. Eleggibilità e gem score

Un comune entra in classifica in un dato mese solo se ha almeno due dei quattro indicatori,
presenze registrate quel mese, overtourism sotto il 75° percentile dell'intero panel e almeno
`MIN_ARRIVI_ANNO` arrivi nell'anno.

La soglia di overtourism è **globale**, non calcolata mese per mese. È una scelta che cambia il
risultato: con una soglia per mese il numero di eleggibili resterebbe piatto attorno a 225 per
costruzione e la stagionalità sparirebbe dalla mappa. Con la soglia globale il numero di comuni in
classifica scende da 186 a febbraio a 102 ad agosto, che è precisamente il fenomeno da mostrare.

```
gem_score = (1 − overtourism) × (attrattività / 100) × (1 + correzione stagionale) × (1 + bonus costiero)
```

In [ ]:
P75 = float(overtourism.indice_overtourism.quantile(OVERTOURISM_Q))

panel['eligible'] = ((panel.n_indicatori_disponibili >= 2)
                     & (panel.presenze_mese > 0)
                     & (panel.indice_overtourism < P75)
                     & (panel.arrivi >= MIN_ARRIVI_ANNO))

panel['gem_score'] = np.where(panel.eligible,
    (1 - panel.indice_overtourism) * (panel.attrattivita / 100)
    * (1 + panel.seasonal_adjustment) * (1 + panel.marine_bonus), 0.0)

SMAX = float(panel.loc[panel.eligible, 'gem_score'].max())
panel['gem_0_100'] = (panel.gem_score / SMAX * 100).round(1)

print(f'soglia overtourism (p{OVERTOURISM_Q:.0%}) = {P75:.4f}   gem_score max = {SMAX:.4f}')
print('\nEleggibili per anno:'); print(panel.groupby('anno').eligible.sum().to_string())

conf = panel.pivot_table(index='mese', columns='anno', values='eligible', aggfunc='sum')
conf.index = MESI
print('\nEleggibili per mese e anno:'); print(conf.to_string())

## 7. Classifiche

Il confronto fra due mesi dello stesso anno mostra il meccanismo: ad agosto la costa è fuori
soglia e restano quasi solo comuni interni; a ottobre rientrano i comuni costieri minori, con la
correzione stagionale al massimo.

In [ ]:
def classifica(anno, mese, n=10):
    q = panel[(panel.anno == anno) & (panel.mese == mese) & panel.eligible]
    return (q.nlargest(n, 'gem_0_100')
             [['comune','gem_0_100','attrattivita','indice_overtourism',
               'seasonal_adjustment','marine_bonus']]
             .round(3).reset_index(drop=True))

for anno, mese in [(2025, 8), (2025, 10), (2022, 8)]:
    print(f'\n=== {MESI[mese-1]} {anno} — {int(panel[(panel.anno==anno)&(panel.mese==mese)].eligible.sum())} comuni in classifica ===')
    print(classifica(anno, mese).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), facecolor='white')
for ax, (anno, mese) in zip(axes, [(2025, 8), (2025, 10)]):
    d = classifica(anno, mese, 12).sort_values('gem_0_100')
    ax.barh(d.comune, d.gem_0_100, color='#B8873F')
    ax.set_xlabel('gem score 0-100'); ax.set_title(f'{MESI[mese-1]} {anno}')
    ax.spines[['top','right']].set_visible(False)
fig.suptitle('Gemme nascoste: stesso anno, due mesi diversi')
fig.tight_layout(); fig.savefig(OUT / 'classifiche.png', dpi=140)

fig2, ax = plt.subplots(figsize=(13, 3.6), facecolor='white')
serie = [int(panel[(panel.anno == y) & (panel.mese == m)].eligible.sum())
         for y in YEARS for m in range(1, 13)]
ax.bar(range(48), serie, color=['#B8873F' if (i % 12) == 7 else '#2C5A64' for i in range(48)])
ax.set_xticks(range(0, 48, 12)); ax.set_xticklabels(YEARS)
ax.set_ylabel('comuni in classifica')
ax.set_title('Quante gemme, mese per mese (in ocra: agosto)')
ax.spines[['top','right']].set_visible(False)
fig2.tight_layout(); fig2.savefig(OUT / 'serie_mensile.png', dpi=140)
print('grafici salvati in', OUT)

## 8. Payload per la mappa

Un solo `data.json`: metadati del modello, geometrie semplificate e, per ogni comune, le sei serie
da 48 valori che la mappa consuma (punteggio, overtourism, correzione stagionale, eleggibilità,
presenze, flag presenze stimate).

In [ ]:
steps = [(y, m) for y in YEARS for m in range(1, 13)]
idx   = {s: i for i, s in enumerate(steps)}
panel['step'] = [idx[(y, m)] for y, m in zip(panel.anno, panel.mese)]
panel = panel.sort_values(['comune','step'])

def geometria(g):
    m = mapping(g.simplify(SIMPLIFY_DEG, preserve_topology=True))
    polys = [m['coordinates']] if m['type'] == 'Polygon' else list(m['coordinates'])
    return [[[[round(x, 4), round(y, 4)] for x, y in ring] for ring in poly] for poly in polys]

comuni = []
for cname, grp in panel.groupby('comune', sort=True):
    g = name_by_key[norm(cname)]
    a = annuale[annuale.comune == cname].set_index('anno')
    comuni.append({
        'n': g, 'i': istat[g], 'pv': prov[g], 'co': int(coastal[g]),
        'sup': round(float(grp.superficie_kmq.dropna().iloc[0]), 1) if grp.superficie_kmq.notna().any() else None,
        'pop': int(grp.popolazione_residente.dropna().iloc[0]) if grp.popolazione_residente.notna().any() else None,
        'attr': {str(y): round(float(a.attrattivita[y]), 1) if y in a.index else 0.0 for y in YEARS},
        'pil':  {str(y): [round(float(a[p + '_n'][y]), 1) if y in a.index else 0.0 for p in PILLARS] for y in YEARS},
        'gem':  [float(v) for v in grp.gem_0_100],
        'ovt':  [round(float(v), 4) for v in grp.indice_overtourism],
        'sea':  [round(float(v), 3) for v in grp.seasonal_adjustment],
        'el':   [int(v) for v in grp.eligible],
        'pre':  [int(v) for v in grp.presenze_mese],
        'est':  [int(v) for v in grp.presenze_stimate],
        'g':    geometria(shapes[g]),
    })

payload = {
    'meta': {
        'anni': YEARS, 'steps': [[y, m] for y, m in steps],
        'p75_overtourism': round(P75, 4), 'gem_score_max': round(SMAX, 4),
        'pilastri': PILLARS, 'pesi': WEIGHTS, 'marine_bonus': MARINE_BONUS,
        'seasonal_max': SEASONAL_MAX, 'min_arrivi_anno': MIN_ARRIVI_ANNO,
        'n_comuni': len(comuni), 'costieri': int(sum(coastal.values())),
        'eleggibili_per_step': [int(panel[panel.step == i].eligible.sum()) for i in range(48)],
        'comuni_con_dati_presenze': {str(y): int(intensita[intensita.anno == y].comune.nunique())
                                     for y in YEARS},
    },
    'comuni': comuni,
}

(OUT / 'data.json').write_text(json.dumps(payload, ensure_ascii=False, separators=(',', ':')),
                               encoding='utf-8')
assert len(payload['comuni']) == 377
assert all(len(c[k]) == 48 for c in comuni for k in ['gem','ovt','sea','el','pre','est'])
print('data.json', round((OUT / 'data.json').stat().st_size / 1024), 'KB —', len(comuni), 'comuni × 48 mesi')

## 9. Mappa HTML

`gemme_nascoste_sardegna.template.html` contiene il segnaposto `__DATA__`; questa cella ci inietta
il JSON e produce un file unico, senza dipendenze di rete: i 377 poligoni sono disegnati in SVG con
una proiezione equirettangolare corretta per la latitudine, senza tile né librerie esterne.

Se il template non è presente accanto al notebook la cella lo dice e si ferma qui: il `data.json`
del passo precedente resta comunque valido.

In [ ]:
TEMPLATE = next((p for p in [OUT / 'gemme_nascoste_sardegna.template.html',
                             Path('gemme_nascoste_sardegna.template.html')] if p.exists()), None)

if TEMPLATE is None:
    print('Template HTML non trovato: salto la generazione della mappa.')
else:
    tpl = TEMPLATE.read_text(encoding='utf-8')
    assert '__DATA__' in tpl
    embedded = json.dumps(payload, ensure_ascii=False, separators=(',', ':')).replace('</', '<\\/')
    html = tpl.replace('__DATA__', embedded)
    dest = OUT / 'gemme_nascoste_sardegna.html'
    dest.write_text(html, encoding='utf-8')

    check = dest.read_text(encoding='utf-8')
    assert '__DATA__' not in check
    assert check.count('<script type="application/json" id="embedded-data">') == 1
    assert 'eleggibili_per_step' in check and 'function color' in check
    assert 'quality_multiplier' not in check
    print('mappa scritta:', dest, '—', round(dest.stat().st_size / 1024), 'KB')

## 10. Verifica finale

Ricalcolo indipendente del punteggio a partire dal payload salvato, per accertare che quello che
finisce nella mappa sia esattamente quello che le celle precedenti hanno calcolato.

In [ ]:
ricalcolo = []
for c in payload['comuni']:
    for s, (y, m) in enumerate(steps):
        if not c['el'][s]:
            continue
        atteso = ((1 - c['ovt'][s]) * (c['attr'][str(y)] / 100)
                  * (1 + c['sea'][s]) * (1 + (MARINE_BONUS if c['co'] else 0.0)) / SMAX * 100)
        ricalcolo.append(abs(atteso - c['gem'][s]))

print('osservazioni eleggibili verificate:', len(ricalcolo))
print('scarto massimo dal ricalcolo: %.3f punti' % max(ricalcolo))
assert max(ricalcolo) < 0.6, 'il punteggio nel payload non coincide con la formula'

vuoti = [i for i in range(48) if payload['meta']['eleggibili_per_step'][i] == 0]
assert not vuoti, f'mesi senza alcun comune in classifica: {vuoti}'

print('\nPASS —', len(payload['comuni']), 'comuni × 48 mesi;',
      sum(payload['meta']['eleggibili_per_step']), 'osservazioni in classifica;',
      'punteggio ricalcolato coerente.')